In [1]:
from pathlib import Path
import sys

# Добавляем папку notebooks/karimov в sys.path для импорта rag_mvp.
PROJECT_DIR = Path.cwd()
if (PROJECT_DIR / "rag_mvp").exists():
    sys.path.insert(0, str(PROJECT_DIR))

from rag_mvp.index import (
    build_bm25_retriever_from_processed,
    build_nomic_dense_retriever_from_processed,
    load_markdown_documents,
    prepare_processed_corpus,
    load_dual_corpus_documents,
    build_raw_document_resolver,
    )

RAW_CORPUS_DIR = PROJECT_DIR / "corpus"
PROCESSED_CORPUS_DIR = PROJECT_DIR / "processed"
INDEX_DIR = PROJECT_DIR / ".rag_index"
BM25_INDEX_PATH = INDEX_DIR / "bm25_index.json"
NOMIC_INDEX_DIR = INDEX_DIR / "nomic_dense"

print(f"Working dir: {PROJECT_DIR}")
print(f"Raw corpus dir: {RAW_CORPUS_DIR}")
print(f"Processed corpus dir: {PROCESSED_CORPUS_DIR}")
print(f"Raw corpus exists: {RAW_CORPUS_DIR.exists()}")

Working dir: c:\Users\ASUS\Array\Соревнования\Хакатон СберПРОСТО\itmo-chemcrow2\notebooks\karimov
Raw corpus dir: c:\Users\ASUS\Array\Соревнования\Хакатон СберПРОСТО\itmo-chemcrow2\notebooks\karimov\corpus
Processed corpus dir: c:\Users\ASUS\Array\Соревнования\Хакатон СберПРОСТО\itmo-chemcrow2\notebooks\karimov\processed
Raw corpus exists: True


# RAG MVP: проверка processed + persistent индекса

Ноутбук для быстрой валидации прототипа:
- raw corpus используется как source of truth для ответа,
- processed corpus используется для retrieval,
- BM25 индекс сохраняется на диск и загружается повторно без лишней переиндексации,
- подготовлен отдельный dense retriever на основе Nomic bi-encoder.

In [2]:
from typing import Any

copied = prepare_processed_corpus(RAW_CORPUS_DIR, PROCESSED_CORPUS_DIR)
print(f"Processed files prepared: {len(copied)}")

processed_documents, raw_docs_by_id = load_dual_corpus_documents(
    raw_corpus_dir=RAW_CORPUS_DIR,
    processed_corpus_dir=PROCESSED_CORPUS_DIR,
    )
documents: Any = [raw_docs_by_id[d.doc_id] for d in processed_documents]
print(f"Загружено raw документов: {len(documents)}")
print("Первые 3 doc_id:", [d.doc_id for d in documents[:3]])

retriever: Any = build_bm25_retriever_from_processed(
    raw_corpus_dir=RAW_CORPUS_DIR,
    processed_corpus_dir=PROCESSED_CORPUS_DIR,
    index_path=BM25_INDEX_PATH,
    )
print("BM25 retriever готов")
print(f"Размер in-memory vector_db: {len(retriever.vector_db)}")
print(f"BM25 index path: {BM25_INDEX_PATH}")

Processed files prepared: 24
Загружено raw документов: 24
Первые 3 doc_id: ['chapter_01_osnovy_reakcionnoy_sposobnosti', 'chapter_02_zashitnye_gruppy', 'chapter_03_vybor_rastvoritelya']
BM25 retriever готов
Размер in-memory vector_db: 24
BM25 index path: c:\Users\ASUS\Array\Соревнования\Хакатон СберПРОСТО\itmo-chemcrow2\notebooks\karimov\.rag_index\bm25_index.json


In [3]:
query = "как выбрать растворитель для реакции SN2"
results: Any = retriever.retrieve(query, top_k=5)

print(f"Query: {query}")
for i, hit in enumerate(results, 1):
    preview: Any = hit.text.replace("\n", " ")[:180]
    print(f"{i}. {hit.doc_id} | score={hit.score:.4f}")
    print(f"   {preview}...")

Query: как выбрать растворитель для реакции SN2
1. chapter_01_osnovy_reakcionnoy_sposobnosti | score=7.0781
   # Глава 1. Основы реакционной способности в органическом синтезе  ## Ключевые понятия Органический синтез опирается на управление распределением электронной плотности в молекуле. Н...
2. chapter_03_vybor_rastvoritelya | score=6.1337
   # Глава 3. Выбор растворителя в органическом синтезе  ## Роль растворителя Растворитель влияет на скорость, селективность и безопасность процесса. Даже при одинаковых реагентах сме...
3. chapter_04_kataliz_perehodnymi_metallami | score=3.5749
   # Глава 4. Катализ переходными металлами  ## Общая идея Палладиевый, никелевый и медный катализ открывают эффективные пути построения C-C и C-N связей. Ключевые стадии каталитическ...
4. chapter_10_strategii_optimizacii | score=2.5634
   # Глава 10. Стратегии оптимизации синтетических методик  ## Логика оптимизации Оптимизация начинается с определения целевой метрики: выход, селективность, скорость или с

In [4]:
# Минимальные sanity-check тесты для MVP

# 1) Корпус должен содержать 10 документов
# assert len(documents) == 10, f"Ожидалось 10 документов, получено: {len(documents)}"

# 2) Размер базы векторов должен совпадать с числом документов
assert len(retriever.vector_db) == len(documents), "vector_db и корпус должны быть одинаковой длины"

# 3) Retrieval с top_k=3 должен вернуть 3 результата
hits_3 = retriever.retrieve("ретросинтетический анализ", top_k=3)
assert len(hits_3) == 3, f"Ожидалось 3 результата, получено: {len(hits_3)}"

# 4) Скоринг должен быть отсортирован по убыванию
assert all(hits_3[i].score >= hits_3[i + 1].score for i in range(len(hits_3) - 1)), "Результаты не отсортированы по score"

# 5) Проверка, что при top_k <= 0 возвращается пустой список
assert retriever.retrieve("любой запрос", top_k=0) == [], "При top_k=0 ожидался пустой список"

print("All MVP sanity checks passed")

All MVP sanity checks passed


In [5]:
# Ручная проверка: поменяйте запрос и перезапустите ячейку
manual_query = "оптимизация выхода реакции и контроль примесей"
manual_hits = retriever.retrieve(manual_query, top_k=3)

print(f"Manual query: {manual_query}")
for i, hit in enumerate(manual_hits, 1):
    print(f"{i}. {hit.doc_id} | score={hit.score:.4f}")

Manual query: оптимизация выхода реакции и контроль примесей
1. chapter_07_stereohimiya | score=8.5941
2. chapter_08_ochistka_i_analiz | score=7.0301
3. chapter_10_strategii_optimizacii | score=6.4265


In [6]:
from rag_mvp.eval_queries import EVAL_QUERIES, flatten_queries

pairs = flatten_queries(EVAL_QUERIES)

TOP_K = 3
hits_top1 = 0
hits_topk = 0
per_doc_total: dict[str, int] = {}
per_doc_top1: dict[str, int] = {}
per_doc_topk: dict[str, int] = {}

for expected_doc_id, q in pairs:
    per_doc_total[expected_doc_id] = per_doc_total.get(expected_doc_id, 0) + 1

    ranked = retriever.retrieve(q, top_k=TOP_K)
    predicted_top1 = ranked[0].doc_id if ranked else None
    predicted_topk = {r.doc_id for r in ranked}

    if predicted_top1 == expected_doc_id:
        hits_top1 += 1
        per_doc_top1[expected_doc_id] = per_doc_top1.get(expected_doc_id, 0) + 1

    if expected_doc_id in predicted_topk:
        hits_topk += 1
        per_doc_topk[expected_doc_id] = per_doc_topk.get(expected_doc_id, 0) + 1

n = len(pairs)
print(f"Всего тестовых запросов: {n}")
print(f"Top-1 exact accuracy: {hits_top1}/{n} = {hits_top1 / n:.2%}")
print(f"Top-{TOP_K} hit rate:    {hits_topk}/{n} = {hits_topk / n:.2%}")
print()

print("По документам:")
for doc_id in sorted(EVAL_QUERIES.keys()):
    total = per_doc_total.get(doc_id, 0)
    top1 = per_doc_top1.get(doc_id, 0)
    topk = per_doc_topk.get(doc_id, 0)
    top1_acc = (top1 / total) if total else 0.0
    topk_acc = (topk / total) if total else 0.0
    print(f"- {doc_id}: Top-1={top1}/{total} ({top1_acc:.0%}), Top-{TOP_K}={topk}/{total} ({topk_acc:.0%})")

# Покажем несколько промахов Top-1 для ручного анализа.
print("\nПримеры промахов Top-1:")
shown = 0
for expected_doc_id, q in pairs:
    ranked = retriever.retrieve(q, top_k=TOP_K)
    predicted_top1 = ranked[0].doc_id if ranked else None
    if predicted_top1 != expected_doc_id:
        print(f"Q: {q}")
        print(f"  expected={expected_doc_id}")
        print(f"  predicted_top1={predicted_top1}")
        print(f"  top_{TOP_K}={[r.doc_id for r in ranked]}")
        shown += 1
        if shown >= 5:
            break

Всего тестовых запросов: 40
Top-1 exact accuracy: 31/40 = 77.50%
Top-3 hit rate:    35/40 = 87.50%

По документам:
- chapter_01_osnovy_reakcionnoy_sposobnosti: Top-1=4/4 (100%), Top-3=4/4 (100%)
- chapter_02_zashitnye_gruppy: Top-1=4/4 (100%), Top-3=4/4 (100%)
- chapter_03_vybor_rastvoritelya: Top-1=3/4 (75%), Top-3=4/4 (100%)
- chapter_04_kataliz_perehodnymi_metallami: Top-1=4/4 (100%), Top-3=4/4 (100%)
- chapter_05_oksidaciya_i_vosstanovlenie: Top-1=2/4 (50%), Top-3=2/4 (50%)
- chapter_06_retrosintez: Top-1=2/4 (50%), Top-3=2/4 (50%)
- chapter_07_stereohimiya: Top-1=3/4 (75%), Top-3=3/4 (75%)
- chapter_08_ochistka_i_analiz: Top-1=3/4 (75%), Top-3=4/4 (100%)
- chapter_09_masshtabirovanie: Top-1=3/4 (75%), Top-3=4/4 (100%)
- chapter_10_strategii_optimizacii: Top-1=3/4 (75%), Top-3=4/4 (100%)

Примеры промахов Top-1:
Q: когда протонная среда благоприятствует механизму SN1
  expected=chapter_03_vybor_rastvoritelya
  predicted_top1=chapter_01_osnovy_reakcionnoy_sposobnosti
  top_3=['chapt

In [7]:
from rag_mvp.eval_queries import EVAL_QUERIES, EVAL_CHUNKS, flatten_queries, flatten_chunks
from rag_mvp.index import build_bm25_retriever_from_eval_chunks

# ===== ГЛАВА (CHAPTERS): целые файлы из corpus =====
print("=" * 70)
print("УРОВЕНЬ 1: CHAPTERS (целые файлы)")
print("=" * 70)

pairs = flatten_queries(EVAL_QUERIES)

def eval_chapters(top_k: int = 3) -> tuple[float, float]:
    top1_hits = 0
    topk_hits = 0
    for expected_doc_id, q in pairs:
        ranked = retriever.retrieve(q, top_k=top_k)
        if ranked and ranked[0].doc_id == expected_doc_id:
            top1_hits += 1
        if any(r.doc_id == expected_doc_id for r in ranked):
            topk_hits += 1
    n = len(pairs)
    return top1_hits / n, topk_hits / n

for k in (1, 3, 5):
    top1, topk = eval_chapters(top_k=k)
    print(f"k={k}: Top-1={top1:.2%}; Top-{k} hit={topk:.2%}")

# ===== CHUNKS: реальные отрывки из текстов =====
print("\n" + "=" * 70)
print("УРОВЕНЬ 2: CHUNKS (отрывки из текстов)")
print("=" * 70)

retriever_chunks = build_bm25_retriever_from_eval_chunks(EVAL_CHUNKS)
triples = flatten_chunks(EVAL_CHUNKS)

def eval_chunks_level(top_k: int = 3) -> tuple[float, float]:
    top1_hits = 0
    topk_hits = 0
    for chunk_id, text, q in triples:
        ranked = retriever_chunks.retrieve(q, top_k=top_k)
        if ranked and ranked[0].doc_id == chunk_id:
            top1_hits += 1
        if any(r.doc_id == chunk_id for r in ranked):
            topk_hits += 1
    n = len(triples)
    return top1_hits / n, topk_hits / n

for k in (1, 3, 5):
    top1, topk = eval_chunks_level(top_k=k)
    print(f"k={k}: Top-1={top1:.2%}; Top-{k} hit={topk:.2%}")

# ===== СРАВНЕНИЕ =====
print("\n" + "=" * 70)
print("ИТОГОВОЕ СРАВНЕНИЕ")
print("=" * 70)
ch_top1, _ = eval_chapters(top_k=1)
ch_top3, _ = eval_chapters(top_k=3)
ch_top5, _ = eval_chapters(top_k=5)
ck_top1, _ = eval_chunks_level(top_k=1)
ck_top3, _ = eval_chunks_level(top_k=3)
ck_top5, _ = eval_chunks_level(top_k=5)

print(f"Chapters: Top-1={ch_top1:.0%}, Top-3={ch_top3:.0%}, Top-5={ch_top5:.0%}")
print(f"Chunks:   Top-1={ck_top1:.0%}, Top-3={ck_top3:.0%}, Top-5={ck_top5:.0%}")

УРОВЕНЬ 1: CHAPTERS (целые файлы)
k=1: Top-1=77.50%; Top-1 hit=77.50%
k=3: Top-1=77.50%; Top-3 hit=87.50%
k=5: Top-1=77.50%; Top-5 hit=92.50%

УРОВЕНЬ 2: CHUNKS (отрывки из текстов)
k=1: Top-1=66.67%; Top-1 hit=66.67%
k=3: Top-1=66.67%; Top-3 hit=91.67%
k=5: Top-1=66.67%; Top-5 hit=91.67%

ИТОГОВОЕ СРАВНЕНИЕ
Chapters: Top-1=78%, Top-3=78%, Top-5=78%
Chunks:   Top-1=67%, Top-3=67%, Top-5=67%


In [3]:
# Nomic dense retriever: build/load, retrieval quality, and persistence sanity checks
import time
from typing import Any

from tqdm.auto import tqdm

from rag_mvp.eval_queries import EVAL_QUERIES, flatten_queries
from rag_mvp.index import (
    load_dual_corpus_documents,
    build_raw_document_resolver,
    )
from rag_mvp.nomic_dense_retriever import NomicDenseRetriever

processed_docs, raw_docs_by_id = load_dual_corpus_documents(
    raw_corpus_dir=RAW_CORPUS_DIR,
    processed_corpus_dir=PROCESSED_CORPUS_DIR,
    )
resolver = build_raw_document_resolver(raw_docs_by_id)

# 1) Cold/warm start check for persistent dense index
print("[1/4] Initializing dense retriever and loading/building index...")
dense = NomicDenseRetriever(
    index_dir=NOMIC_INDEX_DIR,
    matryoshka_dim=256,
    batch_size=8,
    show_progress_bar=True,
    document_resolver=resolver,
    )

t0 = time.perf_counter()
status_1 = dense.build_or_load(processed_docs, force_rebuild=False)
t1 = time.perf_counter()

dense_warm = NomicDenseRetriever(
    index_dir=NOMIC_INDEX_DIR,
    matryoshka_dim=256,
    batch_size=8,
    show_progress_bar=True,
    document_resolver=resolver,
    )
t2 = time.perf_counter()
status_2 = dense_warm.build_or_load(processed_docs, force_rebuild=False)
t3 = time.perf_counter()

print(f"First dense init status:  {status_1} ({t1 - t0:.2f}s)")
print(f"Second dense init status: {status_2} ({t3 - t2:.2f}s)")

# 2) Quick manual retrieval check
print("\n[2/4] Running quick manual retrieval check...")
dense_query = "how to choose solvent for SN2 reaction"
dense_hits = dense_warm.retrieve(dense_query, top_k=5)
print(f"Dense query: {dense_query}")
for i, hit in enumerate(dense_hits, 1):
    preview = hit.text.replace("\n", " ")[:160]
    print(f"{i}. {hit.doc_id} | score={hit.score:.4f}")
    print(f"   {preview}...")

# 3) Retrieval quality check on chapter benchmark
print("\n[3/4] Evaluating dense quality on EVAL_QUERIES...")
pairs = flatten_queries(EVAL_QUERIES)
TOP_K_DENSE = 3
hits_top1_dense = 0
hits_topk_dense = 0

for expected_doc_id, q in tqdm(pairs, desc="Dense eval", unit="query"):
    ranked = dense_warm.retrieve(q, top_k=TOP_K_DENSE)
    pred_top1 = ranked[0].doc_id if ranked else None
    pred_topk = {r.doc_id for r in ranked}

    if pred_top1 == expected_doc_id:
        hits_top1_dense += 1
    if expected_doc_id in pred_topk:
        hits_topk_dense += 1

n_dense = len(pairs)
top1_dense = hits_top1_dense / n_dense
topk_dense = hits_topk_dense / n_dense
print("\nDense quality on EVAL_QUERIES:")
print(f"Top-1 exact accuracy: {hits_top1_dense}/{n_dense} = {top1_dense:.2%}")
print(f"Top-{TOP_K_DENSE} hit rate:    {hits_topk_dense}/{n_dense} = {topk_dense:.2%}")

# 4) ID-only retrieval contract sanity check
print("\n[4/4] Running id-only contract sanity checks...")
id_hits = dense_warm.retrieve_ids("retrosynthetic disconnection strategy", top_k=3)
assert len(id_hits) == 3, "Expected 3 id-only hits"
assert all(id_hits[i][1] >= id_hits[i + 1][1] for i in range(len(id_hits) - 1)), "id-only hits are not sorted"
assert all(doc_id in raw_docs_by_id for doc_id, _ in id_hits), "doc_id from dense index not found in raw corpus"

print("Dense retriever sanity checks passed")

c:\Users\ASUS\Array\Соревнования\Хакатон СберПРОСТО\itmo-chemcrow2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[1/4] Initializing dense retriever and loading/building index...
First dense init status:  loaded (0.00s)
Second dense init status: loaded (0.00s)

[2/4] Running quick manual retrieval check...


<All keys matched successfully>


Dense query: how to choose solvent for SN2 reaction
1. chapter_03_vybor_rastvoritelya | score=0.5048
   # Глава 3. Выбор растворителя в органическом синтезе  ## Роль растворителя Растворитель влияет на скорость, селективность и безопасность процесса. Даже при один...
2. chapter_01_osnovy_reakcionnoy_sposobnosti | score=0.5034
   # Глава 1. Основы реакционной способности в органическом синтезе  ## Ключевые понятия Органический синтез опирается на управление распределением электронной пло...
3. chapter_02_zashitnye_gruppy | score=0.4745
   # Глава 2. Защитные группы в многостадийном синтезе  ## Зачем нужны защитные группы В молекулах с несколькими функциональными группами селективность часто невоз...
4. chapter_05_oksidaciya_i_vosstanovlenie | score=0.4479
   # Глава 5. Окисление и восстановление функциональных групп  ## Окисление спиртов Первичные спирты можно переводить в альдегиды или кислоты в зависимости от окис...
5. chunk_000 | score=0.4431
   ## МОСКОВСКИЙ ГОСУДАРСТВЕННЫЙ УНИВЕРС

Dense eval: 100%|██████████| 40/40 [00:02<00:00, 15.06query/s]


Dense quality on EVAL_QUERIES:
Top-1 exact accuracy: 24/40 = 60.00%
Top-3 hit rate:    33/40 = 82.50%

[4/4] Running id-only contract sanity checks...
Dense retriever sanity checks passed


In [4]:
# Unified detailed comparison: BM25 vs Nomic Dense vs BM25+Dense Rank Fusion
import math
from collections import defaultdict

from tqdm.auto import tqdm
from rag_mvp.eval_queries import EVAL_QUERIES, flatten_queries
from rag_mvp.fusion_retriever import BM25DenseRankFusionRetriever

pairs_cmp = flatten_queries(EVAL_QUERIES)
TOP_K_CMP = 5

def evaluate_retriever(name: str, retr, pairs: list[tuple[str, str]], top_k: int = 5) -> dict:
    top1_hits = 0
    topk_hits = 0
    mrr_sum = 0.0
    ndcg_sum = 0.0
    per_doc_total: dict[str, int] = defaultdict(int)
    per_doc_top1: dict[str, int] = defaultdict(int)
    per_doc_topk: dict[str, int] = defaultdict(int)
    wrong_examples: list[tuple[str, str, list[str]]] = []

    for expected_doc_id, query in tqdm(pairs, desc=f"Eval {name}", unit="query"):
        per_doc_total[expected_doc_id] += 1
        ranked = retr.retrieve(query, top_k=top_k)
        ranked_ids = [r.doc_id for r in ranked]

        rank = None
        for i, doc_id in enumerate(ranked_ids, 1):
            if doc_id == expected_doc_id:
                rank = i
                break

        if ranked and ranked[0].doc_id == expected_doc_id:
            top1_hits += 1
            per_doc_top1[expected_doc_id] += 1

        if rank is not None:
            topk_hits += 1
            per_doc_topk[expected_doc_id] += 1
            mrr_sum += 1.0 / rank
            ndcg_sum += 1.0 / math.log2(rank + 1)

        if rank is None and len(wrong_examples) < 8:
            wrong_examples.append((expected_doc_id, query, ranked_ids))

    n = len(pairs)
    return {
        "name": name,
        "n": n,
        "top1": top1_hits / n,
        "hitk": topk_hits / n,
        "mrr": mrr_sum / n,
        "ndcg": ndcg_sum / n,
        "per_doc_total": dict(per_doc_total),
        "per_doc_top1": dict(per_doc_top1),
        "per_doc_topk": dict(per_doc_topk),
        "wrong_examples": wrong_examples,
    }

fusion_retriever = BM25DenseRankFusionRetriever(
    bm25_retriever=retriever,
    dense_retriever=dense_warm,
    rrf_k=60,
    bm25_weight=1.0,
    dense_weight=1.0,
    candidate_k=20,
    document_resolver=resolver,
    )

bm25_stats = evaluate_retriever("BM25", retriever, pairs_cmp, top_k=TOP_K_CMP)
dense_stats = evaluate_retriever("NomicDense", dense_warm, pairs_cmp, top_k=TOP_K_CMP)
fusion_stats = evaluate_retriever("BM25+DenseRRF", fusion_retriever, pairs_cmp, top_k=TOP_K_CMP)

print("\n=== Unified Metrics (same queries, same top-k) ===")
print(f"Top-k used: {TOP_K_CMP}")
print(f"BM25         -> Top-1={bm25_stats['top1']:.2%}, Hit@{TOP_K_CMP}={bm25_stats['hitk']:.2%}, MRR={bm25_stats['mrr']:.4f}, nDCG@{TOP_K_CMP}={bm25_stats['ndcg']:.4f}")
print(f"NomicDense   -> Top-1={dense_stats['top1']:.2%}, Hit@{TOP_K_CMP}={dense_stats['hitk']:.2%}, MRR={dense_stats['mrr']:.4f}, nDCG@{TOP_K_CMP}={dense_stats['ndcg']:.4f}")
print(f"BM25+DenseRRF-> Top-1={fusion_stats['top1']:.2%}, Hit@{TOP_K_CMP}={fusion_stats['hitk']:.2%}, MRR={fusion_stats['mrr']:.4f}, nDCG@{TOP_K_CMP}={fusion_stats['ndcg']:.4f}")

print("\n=== Per-document Top-1 Comparison ===")
doc_rows = []
for doc_id in sorted(EVAL_QUERIES.keys()):
    total = bm25_stats["per_doc_total"].get(doc_id, 0)
    b = bm25_stats["per_doc_top1"].get(doc_id, 0) / max(total, 1)
    d = dense_stats["per_doc_top1"].get(doc_id, 0) / max(total, 1)
    f = fusion_stats["per_doc_top1"].get(doc_id, 0) / max(total, 1)
    doc_rows.append((doc_id, b, d, f, d - b, f - b))

doc_rows = sorted(doc_rows, key=lambda x: x[5], reverse=True)
for doc_id, b, d, f, delta_db, delta_fb in doc_rows:
    sign_db = "+" if delta_db >= 0 else ""
    sign_fb = "+" if delta_fb >= 0 else ""
    print(
        f"{doc_id}: BM25={b:.0%}, Dense={d:.0%}, Fusion={f:.0%}, "
        f"Δ(Dense-BM25)={sign_db}{delta_db:.0%}, Δ(Fusion-BM25)={sign_fb}{delta_fb:.0%}"
    )

print("\n=== Cases where Fusion wins over BM25 and Dense at Top-1 ===")
shown = 0
for expected_doc_id, query in pairs_cmp:
    bm25_top1 = retriever.retrieve(query, top_k=1)[0].doc_id
    dense_top1 = dense_warm.retrieve(query, top_k=1)[0].doc_id
    fusion_top1 = fusion_retriever.retrieve(query, top_k=1)[0].doc_id

    if fusion_top1 == expected_doc_id and bm25_top1 != expected_doc_id and dense_top1 != expected_doc_id:
        print(f"Q: {query}")
        print(f"  expected: {expected_doc_id}")
        print(f"  bm25_top1: {bm25_top1}")
        print(f"  dense_top1: {dense_top1}")
        print(f"  fusion_top1: {fusion_top1}")
        shown += 1
        if shown >= 8:
            break

Eval BM25+DenseRRF: 100%|██████████| 40/40 [00:02<00:00, 14.08query/s]



=== Unified Metrics (same queries, same top-k) ===
Top-k used: 5
BM25         -> Top-1=77.50%, Hit@5=92.50%, MRR=0.8267, nDCG@5=0.8509
NomicDense   -> Top-1=60.00%, Hit@5=90.00%, MRR=0.7175, nDCG@5=0.7633
BM25+DenseRRF-> Top-1=82.50%, Hit@5=95.00%, MRR=0.8812, nDCG@5=0.8989

=== Per-document Top-1 Comparison ===
chapter_06_retrosintez: BM25=50%, Dense=50%, Fusion=75%, Δ(Dense-BM25)=+0%, Δ(Fusion-BM25)=+25%
chapter_08_ochistka_i_analiz: BM25=75%, Dense=50%, Fusion=100%, Δ(Dense-BM25)=-25%, Δ(Fusion-BM25)=+25%
chapter_09_masshtabirovanie: BM25=75%, Dense=75%, Fusion=100%, Δ(Dense-BM25)=+0%, Δ(Fusion-BM25)=+25%
chapter_10_strategii_optimizacii: BM25=75%, Dense=25%, Fusion=100%, Δ(Dense-BM25)=-50%, Δ(Fusion-BM25)=+25%
chapter_01_osnovy_reakcionnoy_sposobnosti: BM25=100%, Dense=50%, Fusion=100%, Δ(Dense-BM25)=-50%, Δ(Fusion-BM25)=+0%
chapter_03_vybor_rastvoritelya: BM25=75%, Dense=75%, Fusion=75%, Δ(Dense-BM25)=+0%, Δ(Fusion-BM25)=+0%
chapter_05_oksidaciya_i_vosstanovlenie: BM25=50%, Dense

In [5]:
# Compact summary for quick copy/check
print("BM25:", {
    "top1": round(bm25_stats["top1"], 4),
    "hitk": round(bm25_stats["hitk"], 4),
    "mrr": round(bm25_stats["mrr"], 4),
    "ndcg": round(bm25_stats["ndcg"], 4),
})
print("NomicDense:", {
    "top1": round(dense_stats["top1"], 4),
    "hitk": round(dense_stats["hitk"], 4),
    "mrr": round(dense_stats["mrr"], 4),
    "ndcg": round(dense_stats["ndcg"], 4),
})
print("BM25+DenseRRF:", {
    "top1": round(fusion_stats["top1"], 4),
    "hitk": round(fusion_stats["hitk"], 4),
    "mrr": round(fusion_stats["mrr"], 4),
    "ndcg": round(fusion_stats["ndcg"], 4),
})
print("Delta(Dense-BM25):", {
    "top1": round(dense_stats["top1"] - bm25_stats["top1"], 4),
    "hitk": round(dense_stats["hitk"] - bm25_stats["hitk"], 4),
    "mrr": round(dense_stats["mrr"] - bm25_stats["mrr"], 4),
    "ndcg": round(dense_stats["ndcg"] - bm25_stats["ndcg"], 4),
})
print("Delta(Fusion-BM25):", {
    "top1": round(fusion_stats["top1"] - bm25_stats["top1"], 4),
    "hitk": round(fusion_stats["hitk"] - bm25_stats["hitk"], 4),
    "mrr": round(fusion_stats["mrr"] - bm25_stats["mrr"], 4),
    "ndcg": round(fusion_stats["ndcg"] - bm25_stats["ndcg"], 4),
})

BM25: {'top1': 0.775, 'hitk': 0.925, 'mrr': 0.8267, 'ndcg': 0.8509}
NomicDense: {'top1': 0.6, 'hitk': 0.9, 'mrr': 0.7175, 'ndcg': 0.7633}
BM25+DenseRRF: {'top1': 0.825, 'hitk': 0.95, 'mrr': 0.8812, 'ndcg': 0.8989}
Delta(Dense-BM25): {'top1': -0.175, 'hitk': -0.025, 'mrr': -0.1092, 'ndcg': -0.0875}
Delta(Fusion-BM25): {'top1': 0.05, 'hitk': 0.025, 'mrr': 0.0546, 'ndcg': 0.048}
